# Antialiased lines

`Canvas.line()` can draw smooth lines with a width measured in pixels. This guide compares line widths, explains what the aggregate contains, and shows how to control contributions from overlapping parts of a line. All examples use small, generated pandas DataFrames.


In [ ]:
import datashader as ds
import datashader.transfer_functions as tf
import numpy as np
import pandas as pd

x = np.linspace(0, 4 * np.pi, 80)
frame = pd.DataFrame({"x": x, "y": np.sin(x)})
canvas = ds.Canvas(
    plot_width=240, plot_height=100,
    x_range=(0, 4 * np.pi), y_range=(-1.2, 1.2),
)


## Choosing a width

The default `line_width=0` uses a fast, single-pixel line rasterizer. A positive width enables antialiasing: pixels along the edges receive fractional contributions, making diagonal and curved lines appear smoother. Larger widths make the line thicker; widths between zero and one reduce the contribution of a one-pixel-wide line to approximate a thinner line.

Here each image uses the same canvas, colors, and shading span so the effect of changing the width is visible.


In [ ]:
widths = (0, 0.5, 1, 3)
aggregates = [
    canvas.line(frame, "x", "y", agg=ds.count(), line_width=width)
    for width in widths
]
tf.Images(*[
    tf.shade(aggregate, how="linear", span=(0, 1), name=f"line_width={width}")
    for width, aggregate in zip(widths, aggregates)
])


## Interpreting the aggregate

With `ds.count()`, a non-antialiased aggregate contains unsigned integer counts, with zero for empty pixels. An antialiased count aggregate contains floating-point contributions, with `NaN` for empty pixels. The fractional values describe the line's contribution to a pixel; they are not integer counts of observations. Changing the width can change these values as well as the pixels touched by a line.

Inspect the aggregate before shading when those numerical values matter. Shading chooses how values map to colors; it does not change the aggregate.


In [ ]:
[(width, str(aggregate.dtype)) for width, aggregate in zip(widths, aggregates)]


## Overlapping parts of a line

By default, `ds.count()` adds contributions where a line crosses itself. Use `ds.count(self_intersect=False)` to take the maximum contribution from each line at a pixel instead. Contributions from separate lines are still added. This is useful when a line loops back over a pixel but should not appear denser just because of that self-intersection.

The following example draws two lines, one per row (`axis=1`). Each line retraces part of its path. Both images use a fixed shading span so their intensities can be compared.


In [ ]:
paths = pd.DataFrame({
    "x0": [0.0, 0.0], "x1": [1.0, 1.0], "x2": [0.0, 0.0],
    "y0": [0.0, 1.0], "y1": [1.0, 0.0], "y2": [0.0, 1.0],
})
path_canvas = ds.Canvas(
    plot_width=160, plot_height=160,
    x_range=(-0.1, 1.1), y_range=(-0.1, 1.1),
)
path_aggregates = [
    path_canvas.line(
        paths, x=["x0", "x1", "x2"], y=["y0", "y1", "y2"], axis=1,
        agg=ds.count(self_intersect=self_intersect), line_width=5,
    )
    for self_intersect in (True, False)
]
tf.Images(*[
    tf.shade(aggregate, how="linear", span=(0, 4), name=f"self_intersect={value}")
    for value, aggregate in zip((True, False), path_aggregates)
])


## Reductions and performance

Antialiasing is part of line aggregation, so set `line_width` on `Canvas.line()` rather than on `tf.shade()`. Use the [API's reduction table](https://datashader.org/api.html#reductions) to check which reductions support antialiased lines. For example, `ds.sum("value")` aggregates a value column, and `ds.by("category", ds.count())` keeps separate aggregates for categories. The reduction determines how intersecting lines combine; smoothing alone does not define that meaning.

Positive widths require more work than the default rasterizer, especially for thick lines. Start with a width of one and increase it only when needed for visibility. CUDA-backed DataFrames currently fall back to `line_width=0` with a warning, so use a CPU-backed pandas DataFrame for these examples.

The older `antialias=True` argument is equivalent to `line_width=1` and is kept for compatibility. Prefer `line_width` in new code and avoid passing both arguments.

For plotting many curves and choosing aggregators, continue with the [Timeseries guide](https://datashader.org/user_guide/Timeseries.html).
